# Build the EN tool-calling audio dataset — Gemini + Voxtral TTS -> HF

Tourne sur **L4**. Pipeline : generation texte avec **Gemini** -> synthese voix avec
**Voxtral TTS** (vLLM-Omni) -> assemblage `DatasetDict` -> push sur ton HF (prive).

- **Cout** : generation Gemini pour ~3k exemples ~= **$1** (cellule d'estimation, garde-fou < 15 EUR).
- **Licence** : l'audio est synthetise par un modele TTS **CC-BY-NC-4.0** -> le dataset est
  marque non-commercial (recherche). **Garde le repo prive** si tu n'es pas sur de tes droits.
- A renseigner plus bas : `REPO_URL`, `BRANCH`, `HF_REPO`, `GEMINI_API_KEY`, token HF, et les
  **noms de voix** Voxtral.


## 1. Installer les dependances

In [ ]:
import sys, subprocess
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
pip("-U", "vllm", "vllm-omni")                       # serveur Voxtral TTS
pip("google-genai", "httpx", "soundfile", "torchaudio",
    "datasets", "huggingface_hub", "mistral_common")
print("deps ok")

## 2. Recuperer le repo (scripts de generation / TTS / push)

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/Rcarvalo/finetuning_s2s_toolcalling"   # <-- ton repo
BRANCH   = "claude/blissful-tesla-7i1yky"                              # <-- ta branche
WORK     = "/content/finetuning_s2s_toolcalling"

if not os.path.exists(WORK):
    subprocess.run(["git", "clone", REPO_URL, WORK], check=True)
subprocess.run(["git", "-C", WORK, "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "-C", WORK, "checkout", BRANCH], check=True)
subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", WORK], check=True)
os.chdir(WORK); sys.path.insert(0, WORK + "/src"); sys.path.insert(0, WORK + "/scripts")
print("repo:", WORK)

## 3. Cles + parametres

In [ ]:
import os, getpass
os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY") or getpass.getpass("GEMINI_API_KEY: ")
from huggingface_hub import login
login()                                    # token HF (write) pour le push

HF_REPO = "Rcarvalo/tc-en-audio-toolcalling"   # <-- ton dataset (sera prive)
N_TOTAL = 3000                                  # exemples verifies vises
print("HF_REPO:", HF_REPO, "| N_TOTAL:", N_TOTAL)

## 4. Estimation du cout Gemini (garde-fou < 15 EUR)

In [ ]:
# Gemini 2.5 Flash : $0.30 / 1M tokens in, $2.50 / 1M out.
IN_PER_CALL, OUT_PER_CALL = 750, 650            # ~12 cas par appel
calls = int(N_TOTAL / 12 * 1.5)                 # +50% d'overgeneration (rejets/contamination)
cost_usd = calls * IN_PER_CALL / 1e6 * 0.30 + calls * OUT_PER_CALL / 1e6 * 2.50
cost_eur = cost_usd * 0.93
print(f"~{calls} appels Gemini - estimation ~= ${cost_usd:.2f}  (~{cost_eur:.2f} EUR)")
assert cost_eur < 15, "budget > 15 EUR : reduis N_TOTAL ou passe en Flash-Lite"
print("OK, sous budget.")

## 5. Generer les dialogues texte (Gemini) - verifies + anti-contamination

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "scripts/generate_toolcalling_data.py",
    "--provider", "gemini", "--output", "data/tc_en_train.jsonl",
    "--n-total", str(N_TOTAL),
    "--held-out", "benchmark/toolcalling_en/cases.sample.jsonl"], check=True)
print(subprocess.run(["wc", "-l", "data/tc_en_train.jsonl"], capture_output=True, text=True).stdout)

## 6. Lancer Voxtral TTS (vLLM-Omni) en arriere-plan

In [ ]:
import subprocess, time, httpx
LOG = open("/content/voxtral.log", "w")
srv = subprocess.Popen(["vllm", "serve", "mistralai/Voxtral-4B-TTS-2603", "--omni"],
                       stdout=LOG, stderr=subprocess.STDOUT)
for _ in range(180):                            # ~30 min max (download + load)
    try:
        if httpx.get("http://localhost:8000/v1/models", timeout=5).status_code == 200:
            print("Voxtral pret."); break
    except Exception:
        pass
    time.sleep(10)
else:
    raise RuntimeError("serveur Voxtral non pret -- voir /content/voxtral.log")

## 7. Choisir les voix Voxtral
Voxtral fournit ~20 voix preset. Liste les fichiers du modele pour trouver les noms exacts,
puis renseigne `TRAIN_VOICES` / `TEST_VOICES` (held-out = voix de test, idealement disjointes).

In [ ]:
from huggingface_hub import list_repo_files
files = list_repo_files("mistralai/Voxtral-4B-TTS-2603")
print("\n".join(f for f in files if "voice" in f.lower() or f.endswith((".wav", ".json"))))

# <-- remplace par les vrais noms (le test vise des voix INCONNUES du train).
TRAIN_VOICES = ["casual_male"]
TEST_VOICES  = ["casual_male"]

# petit probe : verifie qu'une voix repond bien
import httpx
r = httpx.post("http://localhost:8000/v1/audio/speech", timeout=120,
               json={"input": "Quick test.", "model": "mistralai/Voxtral-4B-TTS-2603",
                     "response_format": "wav", "voice": TRAIN_VOICES[0]})
print("probe voice:", r.status_code, "->", len(r.content), "octets")

## 8. Synthetiser l'audio (train = voix train, benchmark = voix held-out)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "scripts/synthesize_user_audio.py", "--engine", "voxtral",
    "--dialogues", "data/tc_en_train.jsonl", "--audio-root", "data/audio_tc_en",
    "--out", "data/tc_en_train.audio.jsonl", "--split", "train",
    "--voices", ",".join(TRAIN_VOICES)], check=True)
subprocess.run([sys.executable, "scripts/synthesize_user_audio.py", "--engine", "voxtral",
    "--dialogues", "benchmark/toolcalling_en/cases.sample.jsonl", "--audio-root", "data/audio_tc_en",
    "--out", "data/tc_en_bench.audio.jsonl", "--split", "test",
    "--voices", ",".join(TEST_VOICES)], check=True)

## 9. Assembler + pousser le dataset sur ton HF (prive, carte neutre)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "scripts/build_hf_dataset.py", "--repo-id", HF_REPO,
    "--train", "data/tc_en_train.audio.jsonl", "--test", "data/tc_en_bench.audio.jsonl",
    "--audio-root", "data/audio_tc_en", "--private"], check=True)
print("Dataset pousse sur:", f"https://huggingface.co/datasets/{HF_REPO}")

## Suite
Le dataset (audio + cibles tool-call) est pret sur ton HF. Entrainement ensuite avec
`configs/phase_en_toolcalling.yaml` (LoRA backbone, encodeur + tetes audio geles), puis eval
audio via `scripts/eval_audio_toolcalling.py` (cf. README, section *Capacite tool calling vocal EN*).